# BERT Masked Language Modeling 예제

Kaggle Notebook 환경에서 Hugging Face `transformers`를 사용해 `bert-base-uncased` 모델로 `[MASK]` 위치의 단어를 예측합니다.

이 노트북은 다음 흐름으로 구성되어 있습니다.

1. 필요한 라이브러리 설치 및 임포트
2. `AutoTokenizer`와 `AutoModelForMaskedLM` 로드
3. `[MASK]` 토큰이 포함된 문장 입력
4. PyTorch 기반 추론 수행
5. `[MASK]` 위치의 top-5 예측 단어 출력

In [ ]:
# Kaggle 환경에 필요한 패키지가 없을 수 있으므로 먼저 설치합니다.
# 이미 설치되어 있다면 빠르게 넘어갑니다.
%pip install -q transformers torch

In [ ]:
# 기본 라이브러리 임포트
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM

# 실행 장치를 선택합니다. Kaggle에서 GPU를 켜면 cuda가 사용됩니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

## 1. bert-base-uncased 토크나이저와 모델 로드

`AutoTokenizer`는 문장을 BERT가 이해할 수 있는 토큰 ID로 변환합니다. `AutoModelForMaskedLM`은 `[MASK]` 위치의 단어를 예측하는 Masked Language Modeling 헤드를 포함한 BERT 모델입니다.

In [ ]:
# 사용할 사전학습 모델 이름입니다.
model_name = "bert-base-uncased"

# AutoTokenizer와 AutoModelForMaskedLM을 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# 모델을 선택한 장치로 이동하고 평가 모드로 전환합니다.
model.to(device)
model.eval()

print("토크나이저와 BERT MaskedLM 모델 로드 완료")
print(f"mask token    : {tokenizer.mask_token}")
print(f"mask token id : {tokenizer.mask_token_id}")

## 2. [MASK] 토큰이 포함된 문장 입력

`bert-base-uncased`는 영어 소문자 기반 모델입니다. `[MASK]` 토큰은 반드시 토크나이저가 인식하는 형태인 `[MASK]`로 입력합니다.

In [ ]:
# [MASK] 위치의 단어를 BERT가 예측합니다.
text = "The capital of France is [MASK]."

print(text)

## 3. AutoTokenizer로 문장 토큰화

토크나이저는 문장을 `input_ids`, `attention_mask` 등의 텐서로 변환합니다. 이후 `[MASK]` 토큰이 어느 위치에 있는지 찾아야 해당 위치의 예측 결과만 확인할 수 있습니다.

In [ ]:
# 문장을 PyTorch 텐서 형태로 토큰화합니다.
inputs = tokenizer(text, return_tensors="pt")

# 입력 텐서를 모델이 올라간 장치와 동일한 장치로 이동합니다.
inputs = {key: value.to(device) for key, value in inputs.items()}

# 사람이 읽을 수 있도록 토큰 목록을 확인합니다.
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# [MASK] 토큰의 위치를 찾습니다.
mask_positions = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)

# 이 예제는 [MASK]가 하나만 있다고 가정합니다.
mask_batch_index = int(mask_positions[0][0].item())
mask_token_index = int(mask_positions[1][0].item())

print(f"[MASK] 위치: batch={mask_batch_index}, token_index={mask_token_index}")

## 4. PyTorch 기반 추론 수행

`torch.no_grad()`를 사용하면 추론 과정에서 그래디언트를 저장하지 않아 메모리를 아낄 수 있습니다.

In [ ]:
# PyTorch 기반으로 Masked Language Modeling 추론을 수행합니다.
with torch.no_grad():
    outputs = model(**inputs)

# logits shape은 [batch_size, sequence_length, vocab_size]입니다.
logits = outputs.logits
print(f"logits shape: {tuple(logits.shape)}")

# [MASK] 위치의 vocabulary logits만 가져옵니다.
mask_token_logits = logits[mask_batch_index, mask_token_index, :]
print(f"[MASK] logits shape: {tuple(mask_token_logits.shape)}")

## 5. [MASK] 위치의 top-5 예측 단어 출력

`softmax`로 logits를 확률로 바꾼 뒤, 확률이 높은 상위 5개 토큰을 출력합니다.

In [ ]:
# logits를 확률로 변환합니다.
probabilities = torch.softmax(mask_token_logits, dim=-1)

# 확률 기준 상위 5개 토큰을 가져옵니다.
top_k = 5
top_probabilities, top_token_ids = torch.topk(probabilities, k=top_k)

print("[MASK] 위치의 Top-5 예측 결과")
print("-" * 72)
print(f"{'rank':>4} | {'token_id':>8} | {'probability':>11} | token | completed sentence")
print("-" * 72)

for rank, (token_id, probability) in enumerate(zip(top_token_ids.tolist(), top_probabilities.tolist()), start=1):
    predicted_token = tokenizer.decode([token_id]).strip()
    completed_sentence = text.replace(tokenizer.mask_token, predicted_token)
    print(f"{rank:>4} | {token_id:>8} | {probability:>10.4f} | {predicted_token:<10} | {completed_sentence}")

## 다른 문장으로 테스트하기

아래 함수는 `[MASK]`가 하나 포함된 문장을 입력받아 top-k 예측 결과를 반환합니다.

In [ ]:
def predict_masked_words(text, tokenizer, model, device, top_k=5):
    """[MASK]가 하나 포함된 문장에서 top-k 예측 토큰을 반환합니다."""
    if tokenizer.mask_token not in text:
        raise ValueError(f"문장에 {tokenizer.mask_token} 토큰이 포함되어야 합니다.")

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    mask_positions = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
    if len(mask_positions[0]) != 1:
        raise ValueError("이 예제 함수는 [MASK] 토큰이 정확히 하나인 문장만 지원합니다.")

    mask_batch_index = int(mask_positions[0][0].item())
    mask_token_index = int(mask_positions[1][0].item())

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    mask_token_logits = outputs.logits[mask_batch_index, mask_token_index, :]
    probabilities = torch.softmax(mask_token_logits, dim=-1)
    top_probabilities, top_token_ids = torch.topk(probabilities, k=top_k)

    results = []
    for token_id, probability in zip(top_token_ids.tolist(), top_probabilities.tolist()):
        token = tokenizer.decode([token_id]).strip()
        results.append({
            "token_id": token_id,
            "token": token,
            "probability": probability,
            "completed_sentence": text.replace(tokenizer.mask_token, token),
        })

    return results


example_text = "A doctor works in a [MASK]."
results = predict_masked_words(example_text, tokenizer, model, device, top_k=5)

print(example_text)
print("-" * 72)
for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['token']} - {result['probability']:.4f} - {result['completed_sentence']}")